# 04 · CRUD en Neo4j

Grafo `(:User)-[:PUSHED|STARRED|FORKED|...]->(:Repo)`. Driver: `neo4j` (Cypher).

**Justificación:** preguntas de conectividad (repositorio con más usuarios, usuarios que comparten repos) son recorridos de grafo.

In [1]:
import sys, os
sys.path.insert(0, "../src")          # modulos del proyecto
import config
from datetime import datetime, timezone, date
from pprint import pprint

In [2]:
drv = config.neo4j_driver()
drv.verify_connectivity()
def q(cypher, **p):
    recs = drv.execute_query(cypher, database_=config.NEO4J_DATABASE, **p).records
    return [dict(r) for r in recs]
q('MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n')

[{'label': 'User', 'n': 96046},
 {'label': 'Repo', 'n': 116924},
 {'label': 'Org', 'n': 20177}]

## C · Crear nodos y relación

In [3]:
q("MERGE (u:User {login:'estudiante_utec'}) MERGE (r:Repo {name:'utec/bigdata-demo'}) "
  "MERGE (u)-[rel:STARRED]->(r) SET rel.count = 1, rel.first_seen = datetime() RETURN u.login, type(rel), r.name")

[{'u.login': 'estudiante_utec',
  'type(rel)': 'STARRED',
  'r.name': 'utec/bigdata-demo'}]

## R · Consultar

In [4]:
q("MATCH (u:User {login:'estudiante_utec'})-[rel]->(r:Repo) RETURN u.login, type(rel), rel.count, r.name")

[{'u.login': 'estudiante_utec',
  'type(rel)': 'STARRED',
  'rel.count': 1,
  'r.name': 'utec/bigdata-demo'}]

In [5]:
# top 5 repos por usuarios distintos conectados
q('MATCH (u:User)-[]->(r:Repo) RETURN r.name AS repo, count(DISTINCT u) AS usuarios ORDER BY usuarios DESC LIMIT 5')

[{'repo': 'morluto/rea', 'usuarios': 121},
 {'repo': 'sherlock-project/sherlock', 'usuarios': 107},
 {'repo': 'Minireel/HGhub', 'usuarios': 106},
 {'repo': 'meta/astryx', 'usuarios': 103},
 {'repo': 'NotRexed/Hookshot', 'usuarios': 80}]

In [6]:
# usuarios que comparten repos con un usuario dado (2 saltos)
top = q('MATCH (u:User)-[]->(:Repo) RETURN u.login AS login, count(*) AS n ORDER BY n DESC LIMIT 1')
if top:
    login = top[0]['login']
    print(login, '->', q('MATCH (a:User {login:$login})-[]->(r:Repo)<-[]-(b:User) WHERE a<>b RETURN b.login AS colega, count(DISTINCT r) AS repos_en_comun ORDER BY repos_en_comun DESC LIMIT 5', login=login))

github-actions[bot] -> [{'colega': 'dependabot[bot]', 'repos_en_comun': 113}, {'colega': 'renovate[bot]', 'repos_en_comun': 35}, {'colega': 'github-merge-queue[bot]', 'repos_en_comun': 10}, {'colega': 'mergify[bot]', 'repos_en_comun': 10}, {'colega': 'claude[bot]', 'repos_en_comun': 7}]


## U · Actualizar propiedades

In [7]:
q("MATCH (u:User {login:'estudiante_utec'})-[rel:STARRED]->(r:Repo {name:'utec/bigdata-demo'}) "
  "SET rel.count = rel.count + 1, u.rol = 'estudiante' RETURN u.login, u.rol, rel.count")

[{'u.login': 'estudiante_utec', 'u.rol': 'estudiante', 'rel.count': 2}]

## D · Eliminar relación y nodos

In [8]:
q("MATCH (:User {login:'estudiante_utec'})-[rel:STARRED]->(:Repo {name:'utec/bigdata-demo'}) DELETE rel")
q("MATCH (n) WHERE n.login = 'estudiante_utec' OR n.name = 'utec/bigdata-demo' DETACH DELETE n")
q("MATCH (n) WHERE n.login = 'estudiante_utec' OR n.name = 'utec/bigdata-demo' RETURN count(n) AS quedan")

[{'quedan': 0}]

In [9]:
drv.close()